# Healthcare Data Generator — Validation Notebook

Runs a suite of SQL data-quality checks against the Healthcare ODS database:

1. Table inventory & row counts
2. Encounter date-range & gap check (every calendar day should have data)
3. Daily volume sanity (last 10 days)
4. Referential integrity (orphan foreign-key checks)
5. Duplicate primary-key checks
6. NULL checks on key columns
7. Reference data presence (hospitals/departments/date_dim)
8. Patient-location tables & views
9. Diagnosis families (`icd_reference`) and simulated-stay length-of-stay checks
10. Overall PASS/FAIL summary

Safe to run any time (read-only). Can be scheduled to run after each
Healthcare_Data_Generator run, or on-demand via the Fabric Jobs API.


In [ ]:
# ============================================================================
# CELL 1: Configuration & Non-Interactive Connection
# ============================================================================

import importlib
import subprocess
import sys


def _ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Installing missing package: {pip_name}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pip_name], check=False)


for _pip_name, _import_name in [('pyodbc', 'pyodbc'), ('sqlalchemy', 'sqlalchemy')]:
    _ensure_package(_pip_name, _import_name)

import os

FABRIC_SERVER = os.getenv(
    'FABRIC_SERVER',
    '5mx5ymqwo74ezmng6awpg76wx4-f2smxdwfuzzenbkp2vylj5c5ca.database.fabric.microsoft.com'
)
FABRIC_DB = os.getenv('FABRIC_DB', 'Healthcare ODS-63ba7f40-a784-49f0-ae76-387233bc2616')

try:
    import notebookutils  # noqa: F401
    RUNNING_IN_FABRIC = True
except ImportError:
    RUNNING_IN_FABRIC = False


def _get_fabric_access_token():
    # Non-interactive AAD token for the Fabric-provided notebook identity.
    # 'pbi' is the audience Fabric SQL endpoints/warehouses accept.
    for audience in ('pbi', 'https://database.windows.net/'):
        try:
            import notebookutils
            return notebookutils.credentials.getToken(audience)
        except Exception:
            pass
        try:
            import mssparkutils
            return mssparkutils.credentials.getToken(audience)
        except Exception:
            pass
    return None


import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.pool import SingletonThreadPool

print('Connecting to Healthcare ODS...')

if RUNNING_IN_FABRIC:
    token = _get_fabric_access_token()
    raw_odbc_str = (
        f'Driver={{ODBC Driver 18 for SQL Server}};Server=tcp:{FABRIC_SERVER},1433;'
        f'Database={FABRIC_DB};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    )

    def _creator():
        import struct
        token_bytes = token.encode('utf-16-le')
        token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
        return pyodbc.connect(raw_odbc_str, attrs_before={1256: token_struct})

    ENGINE = create_engine('mssql+pyodbc://', creator=_creator, pool_pre_ping=True, poolclass=SingletonThreadPool)
else:
    CONNECTION_STRING = os.getenv('CONNECTION_STRING') or os.getenv('FABRIC_CONNECTION_STRING') or (
        f'Driver={{ODBC Driver 18 for SQL Server}};Server=tcp:{FABRIC_SERVER},1433;'
        f'Database={FABRIC_DB};Authentication=ActiveDirectoryInteractive;'
        f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    )
    ENGINE = create_engine(f'mssql+pyodbc:///?odbc_connect={CONNECTION_STRING}', pool_pre_ping=True, poolclass=SingletonThreadPool)

with ENGINE.connect() as conn:
    conn.execute(text('SELECT 1'))
print('\u2713 Connected.')

CHECK_RESULTS = []  # list of (check_name, passed: bool, detail: str)


def record(check_name, passed, detail=''):
    CHECK_RESULTS.append((check_name, passed, detail))
    icon = '\u2713' if passed else '\u2717'
    print(f'  {icon} {check_name}{": " + detail if detail else ""}')


In [ ]:
# ============================================================================
# CELL 2: Table Inventory & Row Counts
# ============================================================================
import pandas as pd

print('\n[1/8] Table inventory & row counts')
print('-' * 60)

table_counts_df = pd.read_sql('''
    SELECT t.name AS table_name, p.rows AS row_count
    FROM sys.tables t
    JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0,1)
    ORDER BY t.name
''', ENGINE)

TABLE_COUNTS = dict(zip(table_counts_df['table_name'], table_counts_df['row_count']))
print(table_counts_df.to_string(index=False))

expected_tables = [
    'patients', 'doctors', 'encounters', 'diagnoses', 'procedures', 'medications',
    'labs', 'insurance', 'billing', 'admissions', 'hospitals', 'departments',
    'date_dim', 'hospital_department_beds', 'floors', 'rooms', 'beds',
    'patient_bed_assignments'
]
missing_tables = [t for t in expected_tables if t not in TABLE_COUNTS]
record('All expected tables present', len(missing_tables) == 0, f'missing: {missing_tables}' if missing_tables else f'{len(expected_tables)} tables found')


In [ ]:
# ============================================================================
# CELL 3: Encounter Date-Range & Gap Check
# ============================================================================

print('\n[2/8] Encounter date-range & gap check')
print('-' * 60)

date_range_df = pd.read_sql('''
    SELECT MIN(encounter_date) AS min_d, MAX(encounter_date) AS max_d,
           COUNT(DISTINCT CAST(encounter_date AS DATE)) AS distinct_days
    FROM encounters
''', ENGINE)
min_d, max_d, distinct_days = date_range_df.iloc[0]
print(f'Min date: {min_d}   Max date: {max_d}   Distinct days: {distinct_days}')

calendar_df = pd.read_sql('''
    WITH days AS (
        SELECT CAST(encounter_date AS DATE) AS d FROM encounters GROUP BY CAST(encounter_date AS DATE)
    )
    SELECT DATEDIFF(day, MIN(d), MAX(d)) + 1 AS calendar_days, COUNT(*) AS days_with_data FROM days
''', ENGINE)
calendar_days, days_with_data = calendar_df.iloc[0]
missing_days = int(calendar_days) - int(days_with_data)
print(f'Calendar days in range: {calendar_days}   Days with data: {days_with_data}   Missing: {missing_days}')

record('Zero missing calendar days', missing_days == 0, f'{missing_days} missing day(s)')


In [ ]:
# ============================================================================
# CELL 4: Daily Volume Sanity (last 10 days)
# ============================================================================

print('\n[3/8] Daily volume sanity (last 10 days)')
print('-' * 60)

recent_df = pd.read_sql('''
    SELECT TOP 10 CAST(encounter_date AS DATE) AS d, COUNT(*) AS encounters
    FROM encounters GROUP BY CAST(encounter_date AS DATE) ORDER BY d DESC
''', ENGINE)
print(recent_df.to_string(index=False))

low_volume_days = recent_df[recent_df['encounters'] < 100]
record('Recent days have healthy volume (>=100/day)', len(low_volume_days) == 0,
       f'{len(low_volume_days)} day(s) below 100 encounters' if len(low_volume_days) else '')


In [ ]:
# ============================================================================
# CELL 5: Referential Integrity (Orphan Checks)
# ============================================================================

print('\n[4/8] Referential integrity (orphan checks)')
print('-' * 60)

orphan_checks = [
    ('diagnoses -> encounters', 'SELECT COUNT(*) FROM diagnoses d LEFT JOIN encounters e ON d.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
    ('procedures -> encounters', 'SELECT COUNT(*) FROM procedures p LEFT JOIN encounters e ON p.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
    ('medications -> encounters', 'SELECT COUNT(*) FROM medications m LEFT JOIN encounters e ON m.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
    ('labs -> encounters', 'SELECT COUNT(*) FROM labs l LEFT JOIN encounters e ON l.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
    ('encounters -> patients', 'SELECT COUNT(*) FROM encounters e LEFT JOIN patients p ON e.patient_id = p.patient_id WHERE p.patient_id IS NULL'),
    ('billing -> encounters', 'SELECT COUNT(*) FROM billing b LEFT JOIN encounters e ON b.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
    ('billing -> patients', 'SELECT COUNT(*) FROM billing b LEFT JOIN patients p ON b.patient_id = p.patient_id WHERE p.patient_id IS NULL'),
    ('insurance -> patients', 'SELECT COUNT(*) FROM insurance i LEFT JOIN patients p ON i.patient_id = p.patient_id WHERE p.patient_id IS NULL'),
    ('admissions -> encounters', 'SELECT COUNT(*) FROM admissions a LEFT JOIN encounters e ON a.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL'),
]

for label, sql in orphan_checks:
    try:
        orphans = pd.read_sql(sql, ENGINE).iloc[0, 0]
        record(f'{label} (no orphans)', orphans == 0, f'{orphans} orphan(s)' if orphans else '')
    except Exception as e:
        print(f'  \u26A0\ufe0f  {label} skipped: {e}')


In [ ]:
# ============================================================================
# CELL 6: Duplicate Primary Key Checks
# ============================================================================

print('\n[5/8] Duplicate primary key checks')
print('-' * 60)

pk_checks = [
    ('patients', 'patient_id'), ('doctors', 'provider_id'), ('encounters', 'encounter_id'),
    ('diagnoses', 'diagnosis_id'), ('procedures', 'procedure_id'), ('medications', 'medication_id'),
    ('labs', 'lab_id'), ('billing', 'billing_id'), ('insurance', 'insurance_id'), ('admissions', 'admission_id'),
]

for table, pk in pk_checks:
    if table not in TABLE_COUNTS:
        continue
    try:
        dupes = pd.read_sql(f'SELECT COUNT(*) - COUNT(DISTINCT {pk}) FROM {table}', ENGINE).iloc[0, 0]
        record(f'{table}.{pk} unique', dupes == 0, f'{dupes} duplicate(s)' if dupes else '')
    except Exception as e:
        print(f'  \u26A0\ufe0f  {table}.{pk} skipped: {e}')


In [ ]:
# ============================================================================
# CELL 7: NULL Checks & Reference Data
# ============================================================================

print('\n[6/8] NULL checks on key columns')
print('-' * 60)

null_checks = [
    ('patients', 'patient_id'), ('encounters', 'patient_id'),
    ('encounters', 'encounter_date'), ('encounters', 'hospital_id'),
]
for table, col in null_checks:
    if table not in TABLE_COUNTS:
        continue
    try:
        nulls = pd.read_sql(f'SELECT COUNT(*) FROM {table} WHERE {col} IS NULL', ENGINE).iloc[0, 0]
        record(f'{table}.{col} has no NULLs', nulls == 0, f'{nulls} NULL(s)' if nulls else '')
    except Exception as e:
        print(f'  \u26A0\ufe0f  {table}.{col} skipped: {e}')

print('\n[7/8] Reference data presence')
print('-' * 60)
for t in ('hospitals', 'departments', 'date_dim'):
    cnt = TABLE_COUNTS.get(t, 0)
    record(f'{t} populated', cnt > 0, f'{cnt} row(s)')


In [ ]:
# ============================================================================
# ODS quality checks shared by both validation paths
# ============================================================================
# Single source of truth for the diagnosis-family and simulated-stay checks.
#   - scripts/validate_fabric_data.py imports this module directly.
#   - scripts/build_validation_notebook.py embeds this file verbatim into the
#     Fabric validation notebook, which must stay self-contained.
# Keep this file dependency-free (standard library only) so it can be inlined.
#
# Every check's SQL returns a single violation count; 0 means PASS.
# A check whose required tables are absent reports SKIP (never PASS), and a
# query that cannot run reports FAIL rather than being silently ignored.

QUALITY_CHECKS = [
    # --- Diagnosis families (icd_reference) ---------------------------------
    {
        'name': 'Every diagnosis code resolves in icd_reference',
        'requires': ('diagnoses', 'icd_reference'),
        'sql': 'SELECT COUNT(*) FROM dbo.diagnoses d '
               'WHERE NOT EXISTS (SELECT 1 FROM dbo.icd_reference r WHERE r.icd_code = d.icd_code)',
    },
    {
        'name': 'Diagnosis descriptions match icd_reference',
        'requires': ('diagnoses', 'icd_reference'),
        'sql': 'SELECT COUNT(*) FROM dbo.diagnoses d '
               'JOIN dbo.icd_reference r ON r.icd_code = d.icd_code '
               'WHERE d.description IS NULL OR d.description <> r.description',
    },
    {
        'name': 'icd_reference rows are complete and unique',
        'requires': ('icd_reference',),
        'sql': 'SELECT (SELECT COUNT(*) FROM dbo.icd_reference '
               '        WHERE description IS NULL OR clinical_family IS NULL '
               '           OR icd_chapter IS NULL OR chapter_code_range IS NULL) '
               '     + (SELECT COUNT(*) - COUNT(DISTINCT icd_code) FROM dbo.icd_reference)',
    },
    # --- Simulated stays: length-of-stay plan --------------------------------
    # Episodes created before the length-of-stay model carry no plan
    # (target_los_hours IS NULL) and are intentionally excluded.
    {
        'name': 'Planned stay has a complete, self-consistent LOS plan',
        'requires': ('ops_simulated_episode',),
        'sql': 'SELECT COUNT(*) FROM dbo.ops_simulated_episode se '
               'WHERE se.target_los_hours IS NOT NULL AND ('
               '  se.icd_family IS NULL OR se.target_los_hours <= 0 '
               '  OR se.ed_dwell_hours IS NULL OR se.ed_dwell_hours <= 0 '
               '  OR se.icu_expected_flag IS NULL OR se.expected_discharge_datetime IS NULL '
               '  OR ABS(DATEDIFF(SECOND, DATEADD(SECOND, CAST(se.target_los_hours * 3600 AS INT), '
               '         se.created_datetime), se.expected_discharge_datetime)) > 60)',
    },
    {
        'name': 'Planned stay family matches one of its diagnoses',
        'requires': ('ops_simulated_episode', 'diagnoses', 'icd_reference'),
        'sql': 'SELECT COUNT(*) FROM dbo.ops_simulated_episode se '
               'WHERE se.icd_family IS NOT NULL AND NOT EXISTS ('
               '  SELECT 1 FROM dbo.diagnoses d JOIN dbo.icd_reference r ON r.icd_code = d.icd_code '
               '  WHERE d.encounter_id = se.encounter_id AND d.patient_id = se.patient_id '
               '    AND r.clinical_family = se.icd_family)',
    },
    {
        'name': 'No planned stay is clinically discharged before its planned release',
        'requires': ('ops_simulated_episode', 'admissions'),
        'sql': 'SELECT COUNT(*) FROM dbo.ops_simulated_episode se '
               'JOIN dbo.admissions a ON a.admission_id = se.admission_id '
               'WHERE se.expected_discharge_datetime IS NOT NULL AND a.discharge_datetime IS NOT NULL '
               '  AND a.discharge_datetime < DATEADD(MINUTE, -1, se.expected_discharge_datetime)',
    },
    {
        'name': 'Occupied bed carries its stay plan (survives transfers)',
        'requires': ('ops_simulated_episode', 'ops_bed_state'),
        'sql': 'SELECT COUNT(*) FROM dbo.ops_bed_state bs '
               'JOIN dbo.ops_simulated_episode se ON se.encounter_id = bs.encounter_id '
               "WHERE bs.occupancy_status = 'Occupied' AND se.expected_discharge_datetime IS NOT NULL "
               '  AND (bs.expected_release_datetime IS NULL OR ABS(DATEDIFF(SECOND, '
               '       bs.expected_release_datetime, se.expected_discharge_datetime)) > 1)',
    },
    {
        'name': 'ICU transfers only occur for ICU-flagged planned stays',
        'requires': ('ops_simulated_episode', 'ops_patient_movement'),
        'sql': 'SELECT COUNT(*) FROM dbo.ops_patient_movement pm '
               'JOIN dbo.ops_simulated_episode se ON se.encounter_id = pm.encounter_id '
               "WHERE pm.movement_type = 'ICU Transfer' AND se.target_los_hours IS NOT NULL "
               '  AND se.icu_expected_flag = 0',
    },
]

# Informational summaries: printed for review, never PASS/FAIL. Distribution
# targets are statistical and too noisy to gate on small simulated cohorts.
QUALITY_SUMMARIES = [
    {
        'name': 'Simulated length of stay by clinical family',
        'requires': ('ops_simulated_episode',),
        'sql': 'SELECT icd_family, COUNT(*) AS stays, '
               '  CAST(AVG(target_los_hours) / 24 AS DECIMAL(6,2)) AS mean_los_days, '
               '  CAST(AVG(ed_dwell_hours) AS DECIMAL(6,2)) AS mean_ed_dwell_hours, '
               '  CAST(100.0 * AVG(CAST(icu_expected_flag AS FLOAT)) AS DECIMAL(5,1)) AS icu_pct '
               'FROM dbo.ops_simulated_episode WHERE target_los_hours IS NOT NULL '
               'GROUP BY icd_family ORDER BY stays DESC',
    },
    {
        'name': 'Simulated ICU share (published benchmark ~18% of admissions)',
        'requires': ('ops_simulated_episode',),
        'sql': 'SELECT COUNT(*) AS planned_stays, '
               '  CAST(100.0 * AVG(CAST(icu_expected_flag AS FLOAT)) AS DECIMAL(5,1)) AS icu_pct '
               'FROM dbo.ops_simulated_episode WHERE target_los_hours IS NOT NULL',
    },
    {
        'name': 'Diagnoses by clinical family',
        'requires': ('diagnoses', 'icd_reference'),
        'sql': 'SELECT r.clinical_family, COUNT(*) AS diagnoses '
               'FROM dbo.diagnoses d JOIN dbo.icd_reference r ON r.icd_code = d.icd_code '
               'GROUP BY r.clinical_family ORDER BY diagnoses DESC',
    },
]


def run_quality_checks(fetch_rows, existing_tables):
    # fetch_rows(sql) -> list of row tuples; existing_tables: set of table names.
    # Returns a list of (name, status, detail) with status PASS / FAIL / SKIP.
    results = []
    for check in QUALITY_CHECKS:
        missing = [t for t in check['requires'] if t not in existing_tables]
        if missing:
            results.append((check['name'], 'SKIP', 'not deployed: ' + ', '.join(missing)))
            continue
        try:
            violations = int(fetch_rows(check['sql'])[0][0] or 0)
        except Exception as exc:  # noqa: BLE001 - a check that cannot run must fail loudly
            results.append((check['name'], 'FAIL', 'check could not run: ' + str(exc)[:200]))
            continue
        if violations:
            results.append((check['name'], 'FAIL', '{:,} violation(s)'.format(violations)))
        else:
            results.append((check['name'], 'PASS', '0 violations'))
    return results


def run_quality_summaries(fetch_table, existing_tables):
    # fetch_table(sql) -> (column_names, rows). Yields (name, columns, rows) or
    # (name, None, reason) when the summary cannot be produced.
    for summary in QUALITY_SUMMARIES:
        missing = [t for t in summary['requires'] if t not in existing_tables]
        if missing:
            yield summary['name'], None, 'not deployed: ' + ', '.join(missing)
            continue
        try:
            columns, rows = fetch_table(summary['sql'])
        except Exception as exc:  # noqa: BLE001 - informational only
            yield summary['name'], None, 'could not run: ' + str(exc)[:200]
            continue
        yield summary['name'], columns, rows


def format_table(columns, rows, indent='    '):
    text_rows = [[str(c) for c in columns]] + [['' if v is None else str(v) for v in r] for r in rows]
    widths = [max(len(r[i]) for r in text_rows) for i in range(len(columns))]
    lines = [indent + '  '.join(v.ljust(w) for v, w in zip(r, widths)) for r in text_rows]
    lines.insert(1, indent + '  '.join('-' * w for w in widths))
    return '\n'.join(lines)


# ============================================================================
# Diagnosis families & simulated stays
# ============================================================================
print('\nDiagnosis families & simulated stays')
print('-' * 60)

SKIPPED_CHECKS = []


def _fetch_rows(sql):
    return list(pd.read_sql(sql, ENGINE).itertuples(index=False, name=None))


def _fetch_table(sql):
    frame = pd.read_sql(sql, ENGINE)
    return list(frame.columns), list(frame.itertuples(index=False, name=None))


for _name, _status, _detail in run_quality_checks(_fetch_rows, set(TABLE_COUNTS)):
    if _status == 'SKIP':
        # Not deployed is reported, never counted as a pass.
        SKIPPED_CHECKS.append((_name, _detail))
        print(f'  \u2013 {_name}: SKIPPED ({_detail})')
    else:
        record(_name, _status == 'PASS', _detail)

for _name, _columns, _rows in run_quality_summaries(_fetch_table, set(TABLE_COUNTS)):
    print(f'\n  {_name}')
    print(f'    ({_rows})' if _columns is None else format_table(_columns, _rows))


In [ ]:
# ============================================================================
# CELL 8: Views Check & Overall Summary
# ============================================================================

print('\n[8/8] Views check')
print('-' * 60)

views = ['vw_current_er_beds', 'vw_current_patient_beds', 'vw_floor_plan',
         'vw_patient_location', 'vw_floor_occupancy', 'vw_hospital_status']
for view in views:
    try:
        cnt = pd.read_sql(f'SELECT COUNT(*) FROM {view}', ENGINE).iloc[0, 0]
        record(f'VIEW {view} queryable', True, f'{cnt} row(s)')
    except Exception as e:
        record(f'VIEW {view} queryable', False, str(e))

print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)

passed = [c for c in CHECK_RESULTS if c[1]]
failed = [c for c in CHECK_RESULTS if not c[1]]

print(f'Checks passed: {len(passed)}/{len(CHECK_RESULTS)}')
if SKIPPED_CHECKS:
    print(f'Checks skipped (not deployed, not counted as passing): {len(SKIPPED_CHECKS)}')
    for name, detail in SKIPPED_CHECKS:
        print(f'  \u2013 {name}: {detail}')
if failed:
    print('\nFAILED CHECKS:')
    for name, _, detail in failed:
        print(f'  \u2717 {name}: {detail}')
    print('\n\u274c VALIDATION FAILED')
else:
    print('\n\u2705 ALL VALIDATION CHECKS PASSED')
